In [ ]:
%env CUDA_VISIBLE_DEVICES=1
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import os
import torch
from ne_aligned_utils.plot import plot_aligned_embedding
import ot

from ne_aligned_utils.methods import compute_ot_plan
from ne_aligned_utils.utils import sparsify_plan, get_bipartite_graph
from openTSNE.affinity import Affinities

from openTSNE import TSNE

import scipy.sparse as sp

from mpl_toolkits.mplot3d import Axes3D

from ne_aligned_utils.plot import plot_aligned_embedding
from ne_aligned_utils.utils import sparsify_plan
from ne_aligned_utils.data import make_fig1_data

In [ ]:
dataset = "fig1"

tol = 1e-12
maxIter = int(1e4)
res_path = os.path.join("data", dataset)
fig_path = os.path.join("results", "figures")


n0 = 1000
n1 = 5000
colors =  ["r"]*n0 + ["b"]*n1
seeds = [0, 1, 2, 3, 4]
eps_times_list = [1, 10, 100]
embd_dim = 2


In [ ]:
# show data
seed = 1
x0, x1 = make_fig1_data(n0=n0, n1=n1, seed=seed)
# plot in 3d
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(*x0.T, c='r', s=1)
ax.scatter(*x1.T, c='b', s=1)
ax.set_box_aspect([1,1,1])  # Equal aspect ratio

fig.suptitle("Original data")

In [ ]:
eps_times = 1

In [ ]:
P = compute_ot_plan(x0, x1, eps_times=eps_times, tol=tol, max_iter=maxIter)

In [ ]:
P.shape

In [ ]:
U, S, V = np.linalg.svd(P, full_matrices=False)

In [ ]:
n0 = x0.shape[0]
n1 = x1.shape[0]

In [ ]:
t = 0
eot0_embd = np.concatenate([U * S**t * np.sqrt(n0),
                       V.T *S**t * np.sqrt(n1)], axis=0)
t = 1
eot1_embd = np.concatenate([U * S**t * np.sqrt(n0),
                       V.T *S**t * np.sqrt(n1)], axis=0)

t = 0.5
eot_half_embd = np.concatenate([U * S**t * np.sqrt(n0),
                       V.T *S**t * np.sqrt(n1)], axis=0)

In [ ]:
eot0_embd.shape

In [ ]:
S.shape

In [ ]:
P_diag2 = eot0_embd @ np.diag(S) @ eot0_embd.T / np.sqrt(n0 * n1)

In [ ]:
eot_half_embd.shape

In [ ]:
P_diag = eot_half_embd @ eot_half_embd.T / (np.sqrt(n0 * n1))

In [ ]:
np.allclose(P_diag2, P_diag)

In [ ]:
P_diag.shape

In [ ]:
plt.imshow(P_diag, interpolation='none')

In [ ]:
# some entries are neagtive
P_diag[P_diag <0] = 0

In [ ]:
# avoid self-transport
np.fill_diagonal(P_diag, 0)

In [ ]:
# reweigh by size

In [ ]:
1 / P_diag[:n0, :n0].sum(1)[0]

In [ ]:
1 / P_diag[n0:, n0:].sum(1)[0]

In [ ]:
# variants of the graph

In [ ]:
reweigh = np.ones((n0+n1, n0+n1))
reweigh[:n0, :n0] = 1 / P_diag[:n0, :n0].sum(1) / n0
reweigh[n0:, n0:] = 1 / P_diag[n0:, n0:].sum(1) / n1

reweigh /= 4

In [ ]:
P_diag_reweigh = reweigh * P_diag

In [ ]:
P_diag_reweigh[:n0, :n0].sum()

In [ ]:
P_diag_reweigh[:n0, n0:].sum()

In [ ]:
P_diag_reweigh[n0:, :n0].sum()

In [ ]:
P_diag_reweigh[:n0, :n0].sum()

In [ ]:
P_diag_reweigh.sum()

In [ ]:
graph_eq = sp.csr_matrix(P_diag_reweigh)

In [ ]:
graph_no_diag = get_bipartite_graph(P)
graph_no_diag /= graph_no_diag.sum()

In [ ]:
graph = sp.csr_matrix(P_diag/ P_diag.sum())

In [ ]:
init = np.random.randn(n0+n1, 2) * 0.0001

n_epochs = 200

In [ ]:
n_early_epochs = n_epochs // 3
n_late_epochs = n_epochs - n_early_epochs
embedder = TSNE(n_components=embd_dim,
                verbose=True,
                random_state=0,
                early_exaggeration_iter=n_early_epochs,
                n_iter=n_late_epochs)

affinities = Affinities()
affinities.P = graph_no_diag.tocsr()

embd_no_diag = embedder.fit(affinities=affinities, initialization=init)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

colors = ["r"] * n0 + ["b"] * n1

order = np.random.permutation(len(colors))
ax.scatter(*embd_no_diag[order].T, s=1, c=np.array(colors)[order])
ax.set_aspect('equal')
ax.axis("off")

fig.suptitle(f"aligned tSNE without diag (eps={1/eps_times})")
fig.savefig(os.path.join(fig_path, f"aligned_tsne_without_diag_eps_{eps_times}.png"))

In [ ]:
n_early_epochs = n_epochs // 3
n_late_epochs = n_epochs - n_early_epochs
embedder = TSNE(n_components=embd_dim,
                verbose=True,
                random_state=0,
                early_exaggeration_iter=n_early_epochs,
                n_iter=n_late_epochs)

affinities = Affinities()
affinities.P = graph_eq.tocsr()

embd_eq = embedder.fit(affinities=affinities, initialization=init)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

colors = ["r"] * n0 + ["b"] * n1

order = np.random.permutation(len(colors))
ax.scatter(*embd_eq[order].T, s=1, c=np.array(colors)[order])
ax.set_aspect('equal')
ax.axis("off")

fig.suptitle(f"aligned tSNE with eq diag (eps={1/eps_times})")
fig.savefig(os.path.join(fig_path, f"fig1_aligned_tsne_eq_diag_eps_{eps_times}.png"), dpi=300)

In [ ]:
n_early_epochs = n_epochs // 3
n_late_epochs = n_epochs - n_early_epochs
embedder = TSNE(n_components=embd_dim,
                verbose=True,
                random_state=0,
                early_exaggeration_iter=n_early_epochs,
                n_iter=n_late_epochs)

affinities = Affinities()
affinities.P = graph.tocsr()

embd = embedder.fit(affinities=affinities, initialization=init)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

colors = ["r"] * n0 + ["b"] * n1

order = np.random.permutation(len(colors))
ax.scatter(*embd[order].T, s=1, c=np.array(colors)[order])
ax.set_aspect("equal")
ax.axis("off")

fig.suptitle(f"aligned tSNE with diag (eps={1/eps_times})")
fig.savefig(os.path.join(fig_path, f"fig1_aligned_tsne_diag_eps_{eps_times}.png"), dpi=300)

# Sparse

In [ ]:
k = 15

P_diag_sparse = np.zeros_like(P_diag)

P_diag_sparse[:n0, :n0][np.arange(n0,dtype=int)[:, None], P_diag[:n0, :n0].argsort(axis=1)[:, -k:]] = 1
P_diag_sparse[:n0, n0:][np.arange(n0,dtype=int)[:, None], P_diag[:n0, n0:].argsort(axis=1)[:, -int(k*n1/n0):]] = 1

P_diag_sparse[n0:, :n0][np.arange(n1,dtype=int)[:, None], P_diag[n0:, :n0].argsort(axis=1)[:, -k:]] = 1
P_diag_sparse[n0:, n0:][np.arange(n1,dtype=int)[:, None], P_diag[n0:, n0:].argsort(axis=1)[:, -int(k*n1/n0):]] = 1


In [ ]:
P_diag_sparse.sum(1)

In [ ]:
graph_diag_sparse = sp.csr_matrix(P_diag_sparse / P_diag_sparse.sum())
n_early_epochs = n_epochs // 3
n_late_epochs = n_epochs - n_early_epochs
embedder = TSNE(n_components=embd_dim,
                verbose=True,
                random_state=0,
                early_exaggeration_iter=n_early_epochs,
                n_iter=n_late_epochs)

affinities = Affinities()
affinities.P = graph_diag_sparse.tocsr()

embd_diag_sparse = embedder.fit(affinities=affinities, initialization=init)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

colors = ["r"] * n0 + ["b"] * n1

order = np.random.permutation(len(colors))
ax.scatter(*embd_diag_sparse[order].T, s=1, c=np.array(colors)[order])
ax.set_aspect("equal")
ax.axis("off")

fig.suptitle(f"aligned tSNE with sparse diag (eps={1/eps_times})")
fig.savefig(os.path.join(fig_path, f"fig1_aligned_tsne_sparse_diag_eps_{eps_times}.png"), dpi=300)

In [ ]:
P_diag_sparse_sym = np.maximum(P_diag_sparse, P_diag_sparse.T)


In [ ]:
P_diag_sparse_sym[:n0, :n0].sum()/ P_diag_sparse_sym[n0:, :n0].sum()

In [ ]:
P_diag_sparse_sym[n0:, n0:].sum()/ P_diag_sparse_sym[n0:, :n0].sum()


In [ ]:
graph_diag_sparse_sym = sp.csr_matrix(P_diag_sparse_sym / P_diag_sparse_sym.sum())
n_early_epochs = n_epochs // 3
n_late_epochs = n_epochs - n_early_epochs
embedder = TSNE(n_components=embd_dim,
                verbose=True,
                random_state=0,
                early_exaggeration_iter=n_early_epochs,
                n_iter=n_late_epochs)

affinities = Affinities()
affinities.P = graph_diag_sparse_sym.tocsr()

embd_diag_sparse_sym = embedder.fit(affinities=affinities, initialization=init)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

colors = ["r"] * n0 + ["b"] * n1

order = np.random.permutation(len(colors))
ax.scatter(*embd_diag_sparse_sym[order].T, s=1, c=np.array(colors)[order])
ax.set_aspect("equal")
ax.axis("off")

fig.suptitle(f"aligned tSNE with sym sparse diag (eps={1/eps_times})")
fig.savefig(os.path.join(fig_path, f"fig1_aligned_tsne_sym_sparse_diag_eps_{eps_times}.png"), dpi=300)

In [ ]:
k = 15

P_diag_sparse_k = np.zeros_like(P_diag)

P_diag_sparse_k[:n0, :n0][np.arange(n0,dtype=int)[:, None], P_diag[:n0, :n0].argsort(axis=1)[:, -k:]] = 1
P_diag_sparse_k[:n0, n0:][np.arange(n0,dtype=int)[:, None], P_diag[:n0, n0:].argsort(axis=1)[:, -k:]] = 1

P_diag_sparse_k[n0:, :n0][np.arange(n1,dtype=int)[:, None], P_diag[n0:, :n0].argsort(axis=1)[:, -k:]] = 1
P_diag_sparse_k[n0:, n0:][np.arange(n1,dtype=int)[:, None], P_diag[n0:, n0:].argsort(axis=1)[:, -k:]] = 1
print(P_diag_sparse_k.sum(1))
P_diag_sparse_k_sym = np.maximum(P_diag_sparse_k, P_diag_sparse_k.T)


In [ ]:
graph_diag_sparse_k_sym = sp.csr_matrix(P_diag_sparse_k_sym / P_diag_sparse_k_sym.sum())
n_early_epochs = n_epochs // 3
n_late_epochs = n_epochs - n_early_epochs
embedder = TSNE(n_components=embd_dim,
                verbose=True,
                random_state=0,
                early_exaggeration_iter=n_early_epochs,
                n_iter=n_late_epochs)

affinities = Affinities()
affinities.P = graph_diag_sparse_k_sym.tocsr()

embd_diag_sparse_k_sym = embedder.fit(affinities=affinities, initialization=init)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

colors = ["r"] * n0 + ["b"] * n1

order = np.random.permutation(len(colors))
ax.scatter(*embd_diag_sparse_k_sym[order].T, s=1, c=np.array(colors)[order])
ax.set_aspect("equal")
ax.axis("off")

fig.suptitle(f"aligned tSNE with sym sparse-k diag (eps={1/eps_times})")
fig.savefig(os.path.join(fig_path, f"fig1_aligned_tsne_sym_sparse_k_diag_eps_{eps_times}.png"), dpi=300)